In [1]:
import pandas as pd
check = pd.read_csv('messy_sales.csv', encoding='latin-1')

ParserError: Error tokenizing data. C error: Expected 1 fields in line 5, saw 21


## Cleaning plan — messy_sales.csv

Source: messy_sales.csv (10,194 rows × 21 cols as read)
Diagnosed 2026-07-29.

**1. Junk header rows.** 3 lines of export metadata + 1 blank above the real header.
→ `skiprows=4` in `read_csv`. Verified: shape (10194, 21).

**2. Encoding.** File is not valid UTF-8 (product names contain extended chars).
→ `encoding='latin-1'`. Non-negotiable, read fails without it.

**3. Column names.** Four are malformed: `ORDER DATE` (caps), `Customer #` (symbol),
`Sub/Category` (slash), `Postal Code ` (trailing space — invisible, causes KeyError).
→ Normalize all: strip whitespace, lowercase, replace non-alphanumerics with `_`.
Decision: normalize *every* column, not just the four. Consistency beats surgical fixes,
and next month's export will break differently.

**4. Sales stored as text.** dtype `object`. Contains thousands separators (`1,234.50`),
61 values with a `$` prefix, and non-numeric placeholders.
→ Strip `$` and `,` with `.str.replace()`, then `pd.to_numeric(errors='coerce')`.
Count failures BEFORE converting so the log has a real number.

**5. `'-'` placeholders (29 rows).** pandas auto-converts `'N/A'`, `'NA'`, `'null'`, `''`
to NaN, but NOT `'-'`. It survived as literal text in an otherwise-numeric column.
→ Caught by `errors='coerce'` in step 4. Log as "29 non-numeric Sales values coerced to null."
Decision: coerce, don't guess. A dash means "not provided," not zero.

**6. Missing Postal Code (508 rows, 5.0%).**
→ Leave as null. Decision: it's a geographic label, not a measure — nothing downstream
sums or averages it, and imputing a postal code would be fabricating data.

**7. Missing Sales (66 rows, 0.6%).**
→ Leave as NaN; exclude from aggregates via pandas' default skipna behavior.
Decision: do NOT fill with 0. Zero is a real, meaningful value in a sales column — filling
would understate averages and misstate margin. Report the excluded count on the summary sheet.

**8. Exact duplicate rows (200).** Full-row matches, all 21 fields identical.
→ `drop_duplicates(keep='first')`. Defensible without client input: an identical row
cannot be two real transactions.

**9. Ambiguous duplicates (8).** Share Order ID + Product ID but differ elsewhere.
Note: `duplicated()` = 200, `duplicated(subset=['Order ID','Product ID'])` = 208.
The 8 pre-date my corruption — they're in source Superstore.
→ **Do not drop. Flag only.** Decision: could be split shipments, separate line items at
different discounts, or genuine double-entry. Three readings, three different fixes, and
the data can't distinguish them. Requires client confirmation. Listed on the Cleaning Log
sheet as OPEN.

**10. Dates stored as text.** `ORDER DATE` and `Ship Date` are `object`.
→ `pd.to_datetime(errors='coerce')`. Log any that fail to parse.

**11. Numeric-typed labels.** `Row ID` and `Postal Code` appear in `describe()` but are
identifiers, not measures. `Order ID`, `Customer #`, `Product ID` correctly stayed text.
→ No conversion. Decision: never "fix" ID columns to numeric — that's how leading zeros
disappear from account codes.

---

### Expected post-clean state
- Rows: 10,194 → 9,994
- `sales` dtype: float64, with 95 nulls (66 pre-existing + 29 coerced)
- All column names lowercase, underscore-separated, no trailing whitespace
- 8 ambiguous duplicate rows retained and flagged

In [7]:
check = pd.read_csv('messy_sales.csv', encoding='latin-1', skiprows=4)
print(check.shape)

(10194, 21)


In [8]:
check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10194 non-null  int64  
 1   Order ID       10194 non-null  object 
 2   ORDER DATE     10194 non-null  object 
 3   Ship Date      10194 non-null  object 
 4   Ship Mode      10194 non-null  object 
 5   Customer #     10194 non-null  object 
 6   Customer Name  10194 non-null  object 
 7   Segment        10194 non-null  object 
 8   Country        10194 non-null  object 
 9   City           10194 non-null  object 
 10  State          10194 non-null  object 
 11  Postal Code    9686 non-null   float64
 12  Region         10194 non-null  object 
 13  Product ID     10194 non-null  object 
 14  Category       10194 non-null  object 
 15  Sub/Category   10194 non-null  object 
 16  Product Name   10194 non-null  object 
 17  Sales          10128 non-null  object 
 18  Quanti

In [9]:
print(check['Sales'].str.startswith('$').sum())
print((check['Sales'] == '-').sum())

61
29


In [10]:
print(check.duplicated().sum())
print(check.duplicated(subset=['Order ID', 'Product ID']).sum())

200
208


In [11]:
check.describe()

,Row ID,Postal Code,Quantity,Discount,Profit
count,10194.000000,9686.000000,10194.000000,10194.000000,10194.000000
mean,4995.140377,55315.096428,3.788405,0.156657,28.491134
std,2881.410226,32139.594020,2.226518,0.206972,232.258873
min,1.000000,1040.000000,1.000000,0.000000,-6599.978000
25%,2503.250000,23223.000000,2.000000,0.000000,1.726650
50%,4989.500000,59102.000000,3.000000,0.200000,8.635600
75%,7486.750000,90008.000000,5.000000,0.200000,29.321800
max,9994.000000,99301.000000,14.000000,0.800000,8399.976000
